# Open Graph EPV: from a public frame to two team values

**Question:** does player context improve first-shot xG value prediction over ball information?

This offline walkthrough follows a small SkillCorner **training** example through graph construction, future labels, model output, and an illustrative optimization step. It then reads the frozen full-experiment benchmark. The fitting exercise below is single-example memorization, **not** an evaluation or reproduction of the nine trained networks.

Tracking: SkillCorner Open Data, MIT ([license](../third_party/SkillCorner-LICENSE.txt)). Shot estimates: our StatsBomb goal-trained teacher, subject to the [StatsBomb agreement](../third_party/StatsBomb-LICENSE.pdf). [Provenance](../reports/public_provenance.json) records pinned inputs. No private data or pretrained weights are included.

![StatsBomb](../assets/statsbomb-logo.png)


In [1]:
import json
from pathlib import Path
import numpy as np
import torch
from open_graph_epv.data import FEATURES, frame_graph, future_target
from open_graph_epv.team_epv import relative_targets, to_home_away
from open_graph_epv.team_epv_model import TeamGraphEPV

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
example = json.loads((ROOT/'demo/example.json').read_text())
assert example['split'] == 'train'
print('Public training match:', example['match_id'])
print(example['note'])


Public training match: 1874553
Small extract of pinned SkillCorner Open Data. xG and recorded predictions are derived from StatsBomb-trained models; no model weights included.


## 1. Build the graph using information available now

Observed positions and backward velocity form up to 23 nodes. Player IDs align frames but are not model inputs. Invalid players are masked; zero-filled missing coordinates are not players at midfield. Coordinates point towards the possession team's attacking goal. Future shots appear only in the label calculation below.


In [2]:
current, previous, metadata = example['current'], example['previous'], example['metadata']
x, mask, possession_team_id = frame_graph(current, previous, metadata)
print('Features:', FEATURES)
print('Graph shape:', x.shape, '| valid nodes:', int(mask.sum()))
print('Ball features:', np.round(x[0], 3))
assert x.shape == (23, 10) and mask[0]
assert np.isfinite(x).all()


Features: ['x', 'y', 'vx', 'vy', 'teammate', 'opponent', 'ball', 'keeper', 'detected', 'velocity_valid']
Graph shape: (23, 10) | valid nodes: 19
Ball features: [-0.314  0.1   -1.21   0.26   0.     0.     1.     0.     1.     1.   ]


## 2. Define exactly what is predicted

Take the first shot overall in `(t, t + 15 seconds]` within the same period. Assign its estimated xG to the shooting team's label; the other label is zero. If there is no shot, both labels are zero. This is not one future shot independently for each team.

The teacher was fitted separately on goal/non-goal outcomes with geometry, goalkeeper and defender context. This example includes its frozen estimate and provenance, not official StatsBomb xG. Full teacher fitting and the 113 geometry fallbacks are explained in [methods](../docs/methods.md) and [reproduction](../docs/reproduce.md).


In [3]:
period_end = next(p['end_frame'] for p in metadata['match_periods'] if p['period'] == current['period'])
signed_label = future_target(example['future_shots'], current['frame'], current['period'], possession_team_id, period_end)
assert signed_label is not None
relative_label = relative_targets(np.array([signed_label], dtype=np.float32))
possession_is_home = np.array([possession_team_id == metadata['home_team']['id']])
home_away_label = to_home_away(relative_label, possession_is_home)
print('Future shots and teacher source:', example['future_shots'])
print('home_team_target, away_team_target:', home_away_label[0])
print('Possession-relative training label:', relative_label[0])


Future shots and teacher source: [{'period': 1, 'frame': 179, 'team_id': 1802, 'xg': 0.09086118560014837, 'source': 'statsbomb_spatial_context_v1:c750fa6376e994690d5390c7579aed8e03ce2b610dd1da6bc9235773ee38c79c'}]
home_team_target, away_team_target: [0.09086119 0.        ]
Possession-relative training label: [0.09086119 0.        ]


## 3. Forward pass and a small fitting illustration

The GNN exchanges messages between observed nodes, pools team/ball representations, and outputs two conditional expectations. They may both be positive even though one realized window has at most one positive label. A three-coordinate softmax constrains their sum to at most one; unused mass is not a no-shot probability.

MSE targets the conditional mean. The following fresh, small GNN fits **one training frame** for 20 steps to illustrate the API. It cannot establish generalization. It uses neither test labels nor the frozen benchmark weights. No model is saved. The actual benchmark averages three seeds per family, with validation selection and mirror-averaged inference.


In [4]:
torch.manual_seed(0)
torch.set_num_threads(2)
model = TeamGraphEPV(hidden=16, layers=2)
x_tensor = torch.from_numpy(x[None])
mask_tensor = torch.from_numpy(mask[None])
y_tensor = torch.from_numpy(relative_label)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
losses = []
for step in range(20):
    optimizer.zero_grad()
    prediction = model(x_tensor, mask_tensor)
    loss = (prediction - y_tensor).square().mean()
    losses.append(float(loss.detach()))
    loss.backward()
    optimizer.step()
with torch.no_grad():
    fitted = model(x_tensor, mask_tensor).numpy()
print('One-example loss, first/last step:', round(losses[0], 6), round(losses[-1], 6))
print('Illustrative home/away prediction:', to_home_away(fitted, possession_is_home)[0])
assert np.isfinite(fitted).all() and (fitted >= 0).all() and fitted.sum() <= 1.000001
print('This exercise is not a benchmark result.')


One-example loss, first/last step: 0.065566 1.6e-05
Illustrative home/away prediction: [0.11363872 0.00034553]
This exercise is not a benchmark result.


## 4. Read the real, frozen benchmark

The table below comes from full training on 14 matches, selection on 3 validation matches, and evaluation on 3 previously examined test matches. It is independent of the toy fit above. Compare models using the same two-component error metric. The selected family is the **set model**, even though the GNN scores slightly better on test.


In [5]:
results = json.loads((ROOT/'reports/portfolio_results.json').read_text())
print('Validation-selected family:', results['selected_family'])
print(f"{'Model':<12} {'Validation RMSE':>16} {'Test RMSE':>12}")
print(f"{'Prior':<12} {'—':>16} {results['prior']['component_rmse']:12.6f}")
for family in ('ball', 'set', 'gnn'):
    print(f"{family:<12} {results['validation'][family]['component_rmse']:16.6f} {results['test'][family]['component_rmse']:12.6f}")
print('Test snapshots / matches / distinct future shots:', results['n_test_snapshots'], results['n_test_matches'], results['n_distinct_future_shots'])
worst = results['ranked_shot_errors'][0]
print(f"One shot, {worst['n_windows']} windows: {worst['squared_error_share']:.1%} of GNN test squared error")
print(results['status'])


Validation-selected family: set
Model         Validation RMSE    Test RMSE
Prior                       —     0.033899
ball                 0.026231     0.033477
set                  0.025301     0.033305
gnn                  0.025388     0.033200
Test snapshots / matches / distinct future shots: 4228 3 79
One shot, 8 windows: 48.6% of GNN test squared error
Exploratory test on previously examined matches; no causal or native SkillCorner superiority claim.


![Baseline comparison](../reports/figures/portfolio_results.png)

![Example tracking scene](../reports/figures/portfolio_scene.png)

![Error concentration](../reports/figures/portfolio_error_concentration.png)

## 5. Interpretation and next experiment

The small gains do not establish useful tactical understanding. Three test matches and 79 distinct future shots offer limited evidence. Error is concentrated around one high teacher estimate, and transferring an xG teacher between sources introduces uncertainty. The illustrative cases were selected using hindsight.

Before trying another architecture, review high-value labels, missing context and event alignment. Then test on genuinely new matches, retaining a simple baseline. Predicting generated xG is not the same as demonstrating calibrated goal probabilities or causal movement effects.

Explore the recorded predictions in [the offline demo](../demo/index.html). For all nine models and the xG teacher, follow [full reproduction](../docs/reproduce.md). Use [the explanation guide](../docs/explain_the_code.md) to check that you can defend the implementation yourself.
